# 04 — Negatif Kontrol (F3 ADIM 2) 🚦

K-18'in çoğu projede atlanan yarısı. Sisteme **eşleşmemesi gereken** girdiler verilir:

| Girdi | Beklenen |
|---|---|
| Düz yatay çizgi | Pipeline reddeder (σ = 0) |
| White noise | "Eşleşme yok" ya da belirgin kötü skor |
| Whipsaw (testere) | Aynı |
| Tek yönlü rampa | **Tartışmalı — ölçüme bak** |

Ayrıca **skor dağılımı sağlık kontrolü**: tüm sorgular aynı bandı veriyorsa
metrik ayırt etmiyor demektir; bu tek başına F3'ü düşürür.

In [1]:
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))
import numpy as np, pandas as pd, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
pd.set_option("display.width", 200)
OUT = Path("notebooks/output"); OUT.mkdir(parents=True, exist_ok=True)
TF = "1d"

from engine.search.engine import SearchEngine
from tools.f3_measure import flat_line_is_refused, negative_query, query_scores, NEGATIVE_KINDS
from engine.templates import load_all

print("düz yatay çizgi pipeline'da reddediliyor mu?", flat_line_is_refused())
motor = SearchEngine()
dagilim = pd.read_json("data/f3/score_distribution.json")
sablon = dagilim[dagilim.tur == "sablon"].sort_values("en_iyi")
negatif = dagilim[dagilim.tur == "negatif"].copy()
negatif["tip"] = negatif.sorgu.str.rsplit("_", n=1).str[0]
sablon[["sorgu", "sonuc", "en_iyi", "top12", "medyan", "p95"]]

düz yatay çizgi pipeline'da reddediliyor mu? True


,sorgu,sonuc,en_iyi,top12,medyan,p95
1,bear_flag,300,0.008419,0.013341,0.022786,0.028558
2,bull_flag,300,0.009695,0.013627,0.022579,0.028629
5,pennant,300,0.009869,0.016778,0.026427,0.031961
4,falling_wedge,300,0.010563,0.014198,0.020173,0.022956
13,rounding_bottom,300,0.013675,0.022587,0.036828,0.043935
8,cup_and_handle,300,0.014014,0.018389,0.031143,0.037262
6,rising_wedge,300,0.014945,0.021613,0.031598,0.037393
11,head_and_shoulders,300,0.026060,0.031927,0.044998,0.051834
9,double_bottom,300,0.027976,0.048762,0.073878,0.083618
12,inverse_head_and_shoulders,300,0.028264,0.036577,0.049789,0.058119


In [2]:
negatif.groupby("tip")[["sonuc", "en_iyi", "top12", "medyan"]].median().round(4)

,sonuc,en_iyi,top12,medyan
tip,,,,
near_flat,300.0,0.0188,0.0228,0.0324
ramp,300.0,0.0032,0.0054,0.0089
whipsaw,0.0,NaN,NaN,NaN
white_noise,0.0,NaN,NaN,NaN


In [3]:
# Skor dağılımı histogramı — 14 şablon + negatifler
fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(sablon.en_iyi, bins=14, alpha=0.75, label="şablon (14)", color="#2b7a78")
renk = {"white_noise": "#c94f4f", "whipsaw": "#8e44ad", "ramp": "#e08e0b", "near_flat": "#7f8c8d"}
for tip, g in negatif.groupby("tip"):
    sonlu = g.en_iyi[np.isfinite(g.en_iyi)]
    if len(sonlu):
        ax.hist(sonlu, bins=8, alpha=0.6, label=f"negatif: {tip}", color=renk.get(tip))
    else:
        ax.axvline(0, ls=":", color=renk.get(tip), label=f"negatif: {tip} (hiç sonuç yok)")
ax.set_xlabel("en iyi normalize mesafe (küçük = benzer)"); ax.set_ylabel("sorgu sayısı")
ax.set_title("Skor dağılımı — şablon vs negatif kontrol (gerçek veri, 1d)")
ax.legend(); fig.tight_layout(); fig.savefig(OUT / "negatif_kontrol_histogram.png", dpi=110)
print("şablon en iyi skor aralığı: %.4f - %.4f" % (sablon.en_iyi.min(), sablon.en_iyi.max()))
print("white_noise tabanı        : %.4f" % negatif[negatif.tip=='white_noise'].en_iyi.min())

şablon en iyi skor aralığı: 0.0084 - 0.0557
white_noise tabanı        : nan


In [4]:
# Rampa neden negatif sayılmıyor: sonuçları GERÇEKTEN monoton mu?
rng = np.random.default_rng(3)
sonuc = motor.search(negative_query("ramp", rng), timeframe=TF, tolerance="flexible", limit=3)
for m in sonuc.results:
    h = motor.handle(TF, m.window_length)
    satir = int(np.flatnonzero((h.symbol_id == m.symbol_id) & (h.end_idx == m.end_idx))[0])
    v = np.asarray(h.vectors[satir])
    print(f"{m.symbol:10s} L={m.window_length:3d} mesafe={m.distance:.4f} "
          f"genlik=%{m.amplitude:.0f} artan-adım=%{100*np.mean(np.diff(v)>0):.0f}")

ETHUSDT    L= 40 mesafe=0.0027 genlik=%78 artan-adım=%87
TRXUSDT    L= 24 mesafe=0.0029 genlik=%25 artan-adım=%84
SXPUSDT    L= 24 mesafe=0.0035 genlik=%43 artan-adım=%81


**Sonuç:** Rampa sorgusunun döndürdüğü pencereler gerçekten monoton ve yüksek
genlikli. Yani motor doğru cevap veriyor — "trend" gerçek bir şekildir. Bu
yüzden rampa, eşik kalibrasyonunda negatif taban olarak **kullanılmaz**;
kullanılsaydı eşik pozitiflerin altına iner ve motor susardı. Ayrıntı ve açık
soru: `docs/03-f3-dogrulama-raporu.md`.